<a href="https://colab.research.google.com/github/adib422/Sign-Language-Recognition/blob/main/training(On%20colab).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install albumentations onnx onnxruntime opencv-python mediapipe -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.0 MB/s eta 0:00:00


In [1]:
import torch
print(f"PyTorch version: {torch.version}")
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(torch.cuda.get_device_name(0))

PyTorch version: <module 'torch.version' from '/usr/local/lib/python3.12/dist-packages/torch/version.py'>
CUDA available: True
Using device: cuda
Tesla T4


In [2]:
from google.colab import files
import zipfile
import os

print("Please upload your dataset ZIP file (asl_dataset.zip)")
uploaded = files.upload()

# Extract the dataset
zip_filename = list(uploaded.keys())[0]
print(f"Extracting {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('.')

print("✓ Dataset extracted successfully!")
print("Contents:", os.listdir('.'))

Please upload your dataset ZIP file (asl_dataset.zip)


Saving asl_dataset.zip to asl_dataset.zip
Extracting asl_dataset.zip...
✓ Dataset extracted successfully!
Contents: ['.config', 'asl_dataset.zip', 'asl_dataset', 'sample_data']


In [3]:
from google.colab import files

print("Upload dataset_prep.py")
uploaded1 = files.upload()

print("Upload final_app.py (your training script)")
uploaded2 = files.upload()

print("✓ Files uploaded!")

Upload dataset_prep.py


Saving dataset_prep.py to dataset_prep.py
Upload final_app.py (your training script)


Saving final_app.py to final_app.py
✓ Files uploaded!


In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random
import cv2

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2


# CONFIGURATION

DATASET_PATH = "asl_dataset"
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 6

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device('cuda')

# ALBUMENTATIONS TRANSFORMS (GRAYSCALE)

def get_train_transform():
    return A.Compose([
           A.Resize(IMG_SIZE, IMG_SIZE),
           A.Rotate(limit=15, p=0.5),
           A.HorizontalFlip(p=0.3),  # Only keep this
           A.Normalize(mean=[0.5], std=[0.5]),
           ToTensorV2()
       ])


def get_val_transform():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2()
    ])

# DATASET CLASS

class ASLDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        if self.transform:
            transformed = self.transform(image=image)
            image = transformed['image']

        return image, self.labels[idx]

# MAIN PIPELINE

def prepare_data():

    dataset_path = Path(DATASET_PATH)
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset path '{DATASET_PATH}' not found!")

    class_folders = sorted([f for f in dataset_path.iterdir() if f.is_dir()])
    class_to_idx = {f.name: idx for idx, f in enumerate(class_folders)}
    idx_to_class = {idx: name for name, idx in class_to_idx.items()}

    image_paths = []
    labels = []

    for class_folder in tqdm(class_folders, desc="Loading dataset"):
        class_name = class_folder.name
        class_idx = class_to_idx[class_name]

        image_files = list(class_folder.glob("*.jpg")) + \
                     list(class_folder.glob("*.jpeg")) + \
                     list(class_folder.glob("*.png"))

        for img_path in image_files:
            image_paths.append(str(img_path))
            labels.append(class_idx)

    print(f"Total images: {len(image_paths):,}")

    # Split dataset: train/val/test
    X_temp, X_test, y_temp, y_test = train_test_split(
        image_paths, labels, test_size=TEST_RATIO,
        random_state=RANDOM_SEED, stratify=labels
    )

    val_ratio_adj = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_ratio_adj,
        random_state=RANDOM_SEED, stratify=y_temp
    )

    print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

    # Create datasets with Albumentations transforms
    train_dataset = ASLDataset(X_train, y_train, get_train_transform())
    val_dataset = ASLDataset(X_val, y_val, get_val_transform())
    test_dataset = ASLDataset(X_test, y_test, get_val_transform())

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                             shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS)

    print(f"Batch size: {BATCH_SIZE} | Train batches: {len(train_loader)}")

    # Save metadata
    metadata = {
        'class_to_idx': class_to_idx,
        'idx_to_class': idx_to_class,
        'num_classes': len(class_to_idx)
    }
    torch.save(metadata, 'dataset_metadata.pt')

    # Visualize samples
    images, labels_batch = next(iter(train_loader))

    print(f"Image shape: {images.shape} (grayscale)")

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            # Denormalize
            img = images[i] * 0.5 + 0.5
            img = img.squeeze().numpy().clip(0, 1)
            ax.imshow(img, cmap='gray')
            ax.set_title(f"{idx_to_class[labels_batch[i].item()]}")
            ax.axis('off')
    plt.tight_layout()
    plt.savefig('sample_batch.png', dpi=150, bbox_inches='tight')
    print("Saved sample_batch.png")
    plt.close()

    return train_loader, val_loader, test_loader, metadata

if __name__ == "__main__":

    train_loader, val_loader, test_loader, metadata = prepare_data()

    print("\n✅ Dataset Preparartion Complete")


Loading dataset: 100%|██████████| 36/36 [00:01<00:00, 33.89it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Total images: 105,418
Train: 73,792 | Val: 15,813 | Test: 15,813
Batch size: 128 | Train batches: 577
Image shape: torch.Size([128, 1, 224, 224]) (grayscale)
Saved sample_batch.png

✅ Dataset Preparartion Complete


In [5]:
!pip install onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 131.0 MB/s eta 0:00:00


In [7]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.1 MB/s eta 0:00:00


In [9]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 kB 7.0 MB/s eta 0:00:00


In [10]:
# Check all required libraries
import sys

required_packages = {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'numpy': 'numpy',
    'opencv': 'cv2',
    'PIL': 'PIL',
    'sklearn': 'sklearn',
    'albumentations': 'albumentations',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'onnxscript': 'onnxscript',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'tqdm': 'tqdm'
}

print("Checking installed packages...\n")
missing = []

for name, module in required_packages.items():
    try:
        __import__(module)
        print(f"✓ {name}")
    except ImportError:
        print(f"✗ {name} - MISSING")
        missing.append(name)

if missing:
    print(f"\n⚠️ Missing packages: {', '.join(missing)}")
    print(f"\nInstall with:")
    print(f"!pip install {' '.join(missing)}")
else:
    print("\n✓ All required packages installed!")

# Check versions
import torch
import onnx
import onnxruntime
print(f"\nVersions:")
print(f"PyTorch: {torch.__version__}")
print(f"ONNX: {onnx.__version__}")
print(f"ONNX Runtime: {onnxruntime.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Checking installed packages...

✓ torch
✓ torchvision
✓ numpy
✓ opencv
✓ PIL
✓ sklearn
✓ albumentations
✓ onnx
✓ onnxruntime
✓ onnxscript
✓ matplotlib
✓ seaborn
✓ tqdm

✓ All required packages installed!

Versions:
PyTorch: 2.9.0+cu126
ONNX: 1.20.1
ONNX Runtime: 1.23.2
CUDA available: True


In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm

import onnx
import onnxruntime as ort


MODEL_NAME = "mobilenet_v2_grayscale"
NUM_CLASSES = 36
EPOCHS = 15
LEARNING_RATE = 0.001
PATIENCE = 5


OUTPUT_DIR = "output_models"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.pth")
FINAL_MODEL_PATH = os.path.join(OUTPUT_DIR, "final_model.pth")
ONNX_MODEL_PATH = os.path.join(OUTPUT_DIR, "asl_model.onnx")

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device('cuda')
print(f"Using device: {DEVICE}")


class ASLMobileNet(nn.Module):

    def __init__(self, num_classes=36):
        super(ASLMobileNet, self).__init__()

        self.mobilenet = models.mobilenet_v2(pretrained=True)

        original_conv = self.mobilenet.features[0][0]
        self.mobilenet.features[0][0] = nn.Conv2d(
            1,  # Input: 1 channel (grayscale)
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )

        in_features = self.mobilenet.classifier[1].in_features
        self.mobilenet.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.mobilenet(x)


def build_model(num_classes):

    model = ASLMobileNet(num_classes=num_classes)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Model: MobileNetV2 (Grayscale)")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Input: [B, 1, 224, 224] (grayscale)")
    print(f"Output: [B, {num_classes}]")

    return model



# TRAINING
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                         'acc': f'{100.*correct/total:.2f}%'})

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validation", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def train_model(model, train_loader, val_loader, epochs, lr, patience, device):
    print(f"\n{'='*60}")
    print("TRAINING MODEL")
    print(f"{'='*60}")
    print(f"Epochs: {epochs} | LR: {lr} | Patience: {patience}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )

    best_val_acc = 0.0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    start_time = time.time()

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 60)

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        scheduler.step(val_acc)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"✓ Best model saved! (Val Acc: {val_acc:.2f}%)")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    total_time = time.time() - start_time
    print(f"\nTraining completed in {total_time/60:.2f} minutes")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    return model, history



# EVALUATION

def evaluate_model(model, test_loader, device, idx_to_class):
    print(f"\n{'='*60}")
    print("EVALUATION")
    print(f"{'='*60}")

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing"):
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\n✓ Test Accuracy: {accuracy*100:.2f}%")

    class_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
    print(f"\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)
    return accuracy, cm, class_names, all_preds, all_labels


# ONNX EXPORT
def export_to_onnx(model, onnx_path, input_shape=(1, 1, 224, 224)):
    print(f"\n{'='*60}")
    print("EXPORTING TO ONNX")
    print(f"{'='*60}")

    model.eval()

    dummy_input = torch.randn(input_shape)
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=12,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    )

    print(f"✓ ONNX model saved to: {onnx_path}")

    # Verify ONNX model
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX model verified successfully")

    # Get model size
    model_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"✓ ONNX model size: {model_size_mb:.2f} MB")

    return onnx_path


def test_onnx_inference(onnx_path, test_loader, idx_to_class):

    print(f"\n{'='*60}")
    print("TESTING ONNX INFERENCE")
    print(f"{'='*60}")

    # Load ONNX model
    ort_session = ort.InferenceSession(onnx_path)

    # Test on a few batches
    correct = 0
    total = 0
    inference_times = []

    for i, (images, labels) in enumerate(test_loader):
        if i >= 10:  # Test on 10 batches
            break

        # Prepare input
        ort_inputs = {ort_session.get_inputs()[0].name: images.numpy()}

        # Inference
        start_time = time.time()
        ort_outputs = ort_session.run(None, ort_inputs)
        inference_time = (time.time() - start_time) * 1000  # ms
        inference_times.append(inference_time)

        # Get predictions
        predictions = np.argmax(ort_outputs[0], axis=1)

        correct += (predictions == labels.numpy()).sum()
        total += labels.size(0)

    accuracy = 100. * correct / total
    avg_inference_time = np.mean(inference_times)

    print(f"✓ ONNX Accuracy (10 batches): {accuracy:.2f}%")
    print(f"✓ Avg inference time: {avg_inference_time:.2f} ms/batch")
    print(f"✓ Avg inference per image: {avg_inference_time/32:.2f} ms")



# VISUALIZATION

def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history['train_acc'], label='Train', linewidth=2)
    ax1.plot(history['val_acc'], label='Validation', linewidth=2)
    ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy (%)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['train_loss'], label='Train', linewidth=2)
    ax2.plot(history['val_loss'], label='Validation', linewidth=2)
    ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150)
    print(f"Saved training_history.png")
    plt.close()


def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('True', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
    print(f"Saved confusion_matrix.png")
    plt.close()


# MAIN PIPELINE

def main():
    print("\n" + "="*60)
    print("ASL SIGN LANGUAGE DETECTION")
    print("="*60)

    if not os.path.exists('dataset_metadata.pt'):
        raise FileNotFoundError("Run Phase 1&2 first to create dataset_metadata.pt")

    metadata = torch.load('dataset_metadata.pt')
    idx_to_class = metadata['idx_to_class']
    num_classes = metadata['num_classes']

    print(f"Loaded metadata: {num_classes} classes")

    import sys
    sys.path.insert(0, '.')
    from dataset_prep import prepare_data

    train_loader, val_loader, test_loader, _ = prepare_data()

    model = build_model(num_classes)
    model = model.to(DEVICE)


    # Train model
    model, history = train_model(
        model, train_loader, val_loader,
        EPOCHS, LEARNING_RATE, PATIENCE, DEVICE
    )

    torch.save(model.state_dict(), FINAL_MODEL_PATH)
    print(f"\n✓ PyTorch model saved: {FINAL_MODEL_PATH}")

    # Evaluate
    accuracy, cm, class_names, all_preds, all_labels = evaluate_model(
        model, test_loader, DEVICE, idx_to_class
    )

    # Plot results
    print(f"\nGenerating visualizations...")
    plot_training_history(history)
    plot_confusion_matrix(cm, class_names)

    # Export to ONNX
    onnx_path = export_to_onnx(model, ONNX_MODEL_PATH, input_shape=(1, 1, 224, 224))

    # Test ONNX inference
    test_onnx_inference(onnx_path, test_loader, idx_to_class)

    # Save results
    results = {
        'test_accuracy': accuracy,
        'num_classes': num_classes,
        'model_name': MODEL_NAME,
        'idx_to_class': idx_to_class,
        'history': history,
        'onnx_path': onnx_path
    }
    torch.save(results, os.path.join(OUTPUT_DIR, 'results.pt'))

    print(f"\n{'='*60}")
    print("✓ TRAINING COMPLETE!")
    print(f"{'='*60}")
    print(f"\nFinal Results:")
    print(f"  Test Accuracy: {accuracy*100:.2f}%")
    print(f"  PyTorch model: {FINAL_MODEL_PATH}")
    print(f"  ONNX model: {ONNX_MODEL_PATH}")
    print(f"\nFiles in '{OUTPUT_DIR}/':")
    print(f"  - asl_model.onnx (ONNX format - use this for inference!)")
    print(f"  - best_model.pth (PyTorch checkpoint)")
    print(f"  - final_model.pth (PyTorch final)")
    print(f"  - results.pt (metrics)")
    print(f"  - training_history.png")
    print(f"  - confusion_matrix.png")


if __name__ == "__main__":
    main()

Using device: cuda

ASL SIGN LANGUAGE DETECTION
Loaded metadata: 36 classes


Loading dataset: 100%|██████████| 36/36 [00:01<00:00, 20.86it/s]


Total images: 105,418
Train: 73,792 | Val: 15,813 | Test: 15,813
Batch size: 128 | Train batches: 577


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Image shape: torch.Size([128, 1, 224, 224]) (grayscale)


...

/tmp/ipython-input-882405422.py:232: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0112 12:13:43.429000 541 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Saved confusion_matrix.png

EXPORTING TO ONNX
[torch.onnx] Obtain model graph for `ASLMobileNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ASLMobileNet([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `ASLMobileNet([...]` with `torch.export.export(..., strict=True)`...


E0112 12:13:46.424000 541 torch/export/_trace.py:1142] always_classified is unsupported.
E0112 12:13:46.427000 541 torch/export/_trace.py:1142] always_classified is unsupported.


[torch.onnx] Obtain model graph for `ASLMobileNet([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'RuntimeError'>: Tensor on device cuda:0 is not on the expected device cpu!

(Refer to the full stack trace above for more information.)

In [15]:
OUTPUT_DIR = "output_models"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.pth")
ONNX_MODEL_PATH = os.path.join(OUTPUT_DIR, "asl_model.onnx")

NUM_CLASSES = 36

# ===================== MODEL =====================
class ASLMobileNet(nn.Module):
    def __init__(self, num_classes=36):
        super().__init__()

        self.mobilenet = models.mobilenet_v2(pretrained=False)

        original_conv = self.mobilenet.features[0][0]
        self.mobilenet.features[0][0] = nn.Conv2d(
            1,
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )

        in_features = self.mobilenet.classifier[1].in_features
        self.mobilenet.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.mobilenet(x)

# ===================== LOAD TRAINED MODEL =====================
print("Loading trained model...")
model = ASLMobileNet(NUM_CLASSES)

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location="cpu"))
model.eval()

# ===================== EXPORT TO ONNX =====================
print("Exporting to ONNX...")

dummy_input = torch.randn(1, 1, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_MODEL_PATH,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("✓ ONNX export successful:", ONNX_MODEL_PATH)

# ===================== VERIFY =====================
onnx_model = onnx.load(ONNX_MODEL_PATH)
onnx.checker.check_model(onnx_model)
print("✓ ONNX model verified")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipython-input-608380527.py:48: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0112 12:20:51.859000 541 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic

Loading trained model...
Exporting to ONNX...
[torch.onnx] Obtain model graph for `ASLMobileNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ASLMobileNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 104 of general pattern rewrite rules.
✓ ONNX export successful: output_models/asl_model.onnx
✓ ONNX model verified


In [16]:
import shutil

shutil.make_archive(
    "asl_outputs",
    "zip",
    "output_models"
)

print("Zip created: asl_outputs.zip")
from google.colab import files
files.download("asl_outputs.zip")


Zip created: asl_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>